In [ ]:
import requests
from requests.packages.urllib3.util.retry import Retry
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
import csv
from tqdm import tqdm

In [ ]:
class TimeoutHttpAdapter(HTTPAdapter):
    def __init__(self, timeout=None, *args, **kwargs):
        self.timeout = timeout
        if "timeout" in kwargs:
            del kwargs["timeout"]
        super().__init__(*args, **kwargs)

    def send(self, *args, **kwargs):
        kwargs['timeout'] = self.timeout
        return super().send(*args, **kwargs)

In [ ]:
r=requests.Session()
retry_strategy = Retry(
            total=5,
            status_forcelist=[104, 429, 500, 502, 503, 504],
            allowed_methods=["HEAD", "GET" "POST", "PUT", "DELETE", "OPTIONS", "TRACE"],
            backoff_factor=2
        )
r.headers["User-Agent"]='My User Agent 2.0'
r.mount('https://', TimeoutHttpAdapter(timeout=None, max_retries=retry_strategy))
r.mount('http://', TimeoutHttpAdapter(timeout=None, max_retries=retry_strategy))

req = r.get('https://www.lawyersclubindia.com/forum/')

In [ ]:
soup = BeautifulSoup(req.content, 'html.parser')

In [ ]:
s = soup.find_all('div', class_='flex-grow-1 ms-3')

In [ ]:
link_list=[]
for li in s:
    a = li.find("a")
    if a:
        link_list.append(a.attrs["href"])


In [ ]:
link_list


In [ ]:
category_list=[]
for link in link_list:
    if ('cat_id') in link:
        category_list.append('https://www.lawyersclubindia.com/forum/'+link)

In [ ]:
rows=[]
for link in tqdm(category_list):
    temp=link
#     cnt=0
    while(True):
        req=r.get(temp)
        soup = BeautifulSoup(req.content.decode(), "html.parser")
#         print(soup)
        query_link=(soup.find_all('a',class_='text-dark'))
        for qlink in query_link:
            qreq=r.get('https://www.lawyersclubindia.com/forum/'+qlink.attrs['href'])
            qsoup=BeautifulSoup(qreq.content.decode(), "html.parser")
            qthread=qsoup.find_all('div',class_='img-res ft-page-content fluid-column dont-break-out')
            qheading=''
            if qsoup.find('a',class_='text-dark').text is not None:
                qheading=qsoup.find('a',class_='text-dark').text
#             qthread=qthread[:2]
            query_thread=[]
            for item in qthread:
                if(item.find('p') is not None):
                    paragraph=''
                    for para in item.find_all('p'):
                        paragraph+=para.text
                    query_thread.append(paragraph)
            max_len_para=''
            if len(query_thread)>=2:
                for i in range(1,len(query_thread)):
                    if len(query_thread[i])>len(max_len_para):
                        max_len_para=query_thread[i]
                rows.append([qheading.strip(),query_thread[0].strip(),max_len_para.strip()])
#             print(max_len_para.strip())
#             print('***********************************************************')
        nxtpage = (soup.find_all("a", class_="page-link"))[-1]
#         print(nxtpage)
        if nxtpage is None:
            break
        temp='https://www.lawyersclubindia.com/forum/'+nxtpage.attrs['href']
        print(temp)
    

In [ ]:
len(rows)

In [ ]:
fields = ['title', 'question', 'answer'] 
with open('legal_queries_lawyersclub.csv', 'w+') as f:
      
    # using csv.writer method from CSV package
    write = csv.writer(f)
      
    write.writerow(fields)
    write.writerows(rows)

In [ ]:
f.close()